In [0]:
# set whether normal process or being run by Elsevier supporters
permissions = 'fulldata' # default or fulldata
# set the project name
project_name = 'proj_050_teamscience'  # This is the folder name that you work in, e.g. 'test_project_1'

In [0]:
#Alick: I'll switch to the latest snapshot (v008) - wording there is confusing! 
#%run /Snippets/header_latest

In [0]:
%run /Snippets/header_008

## Python Libraries and Functions

In [0]:
# libraries 
from pyspark.sql.functions import col, count, explode, array, mean, stddev, filter, percentile_approx, arrays_zip, size, array_distinct, sum, reverse, collect_list, array_sort, posexplode, max, arrays_overlap, lit, when, cast
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import Window
from pyspark.ml.feature import StringIndexer

# make a DF to store everything
model_output_df = pd.DataFrame(columns = ["model", "coef", "se", "t", "p", "r2"])
descriptives_output_df = pd.DataFrame(columns = ["model", "subjareas", "mean", "sd", "sample", "median", "iqr_25", "iqr_75"])

# model summary print out 
def modelsummary(model, hyp_model, df):
    import numpy as np
    print ("Note: the last rows are the information for Intercept")
    print ("##","-------------------------------------------------")
    print ("##","  Estimate   |   Std.Error | t Values  |  P-value")
    coef = np.append(list(model.coefficients),model.intercept)
    Summary=model.summary

    for i in range(len(Summary.pValues)):
        print ("##",'{:10.6f}'.format(coef[i]),\
        '{:10.6f}'.format(Summary.coefficientStandardErrors[i]),\
        '{:8.3f}'.format(Summary.tValues[i]),\
        '{:10.6f}'.format(Summary.pValues[i]))
        # store the data 
        row = pd.DataFrame({"model": hyp_model, "coef": coef[i], "se": Summary.coefficientStandardErrors[i], 
               "t": Summary.tValues[i], "p": Summary.pValues[i], "r2": Summary.r2}, index = [0])
        df = pd.concat([df, row], ignore_index = True, axis = 0)

    print ("##",'---')
    print ("##","Mean squared error: % .6f" \
           % Summary.meanSquaredError, ", RMSE: % .6f" \
           % Summary.rootMeanSquaredError )
    print ("##","Multiple R-squared: %f" % Summary.r2, ", \
            Total iterations: %i"% Summary.totalIterations)
    return(df)

PROJECT DETAILS:
Project identifier: proj_050_teamscience
Cluster data access level: fulldata


# Research Question 2

What characteristics of team composition have the greatest influence on article metrics? 

Author independent variables: Author characteristics include affiliation country, affiliation/institution, h-index, gender, and gender ratio. We will transform the affiliation country into a count variable of the number of unique countries. Similarly, we will transform the affiliation/institutions into a count of the number of different affiliations across authors. We will use the h-index of the first and last authors for each article. We will use h-index with and without self-citation if this metric is available through the ICSR system. Last, Elsevier has noted that their inferred gender column has a 97% accuracy, and we will use inferred gender of the first and last authors for each article. From this variable, we will additionally calculate the gender ratio (percent male) as a single variable for each article.

In [0]:
df_ani = spark.read.format("parquet").load(basePath+tablename_ani)
df_ani_smaller = df_ani.filter(df_ani.Year > 1969)\
  .filter(df_ani.publication_type == "ar")\
  .sample(False, .25, 8675309) 

df_fwci = spark.read.format("parquet").load(fwci_parquet_path)

In [0]:
# calculate variables, load tables
# gender
df_gender = spark.read.format("parquet").load(basePath+tablename_gender_inference)

# gender ratio
gender_summary = df_ani_smaller.select("Eid", "Au.auid")\
  .withColumn("auid", explode("auid"))\
  .join(df_gender.select("auid", "Inferred_Probable_Gender"), "auid", "left")\
  .groupBy("Eid", "Inferred_Probable_Gender")\
  .agg(count("auid").alias("author_count"))\
  .withColumn("gender_percent", col("author_count")/sum("author_count").over(Window.partitionBy("Eid")))\
  .withColumn("team_size", sum("author_count").over(Window.partitionBy("Eid")))\
  .filter(col("Inferred_Probable_Gender") == "male")\
  .drop("Inferred_Probable_Gender", "author_count")

# display(gender_summary)

In [0]:
# merge all data needed
df_hyp2 = df_ani_smaller.select("Eid", "Year", "Au.Authorseq", "Af.affiliation_country", "Af.affiliation_ids", "ASJC", "CitationCount", "Au.auid")\
  .withColumn("country_count", size(array_distinct("affiliation_country")))\
  .withColumn("inst_count", size(array_distinct("affiliation_ids")))\
  .withColumn("auth_combo", arrays_zip("Authorseq", "auid"))\
  .withColumn("auth_combo", explode("auth_combo"))\
  .select("Eid", "Year", "country_count", "inst_count", "ASJC", "CitationCount", col("auth_combo.Authorseq").alias("Authorseq"), col("auth_combo.auid").alias("auid"))\
  .join(df_gender.select("auid", "Inferred_Probable_Gender"), "auid", "left")\
  .join(df_fwci.select("Eid", "Citations_5y"), "Eid", "left")\
  .join(gender_summary, "Eid", "left")
  
# display(df_hyp2)

In [0]:
# calculate h-index for only those we need
df_authors = df_hyp2.select("Authorseq", "auid", "team_size")\
	.filter((col("Authorseq") == 1) | (col("Authorseq") == col("team_size")))


In [0]:
windowspec = Window.partitionBy('auid').orderBy(func.col('citation_count').desc())

df_hindex =(df_ani
            .select('eid', func.col('Year').alias('year_of_citation'),func.array_distinct('references').alias('references'))
            .withColumn('cited_document', func.explode('references'))
            .join(df_ani.select(func.col('Eid').alias('cited_document'),
                                func.col('Year').alias('cited_doc_pub_year'),
                                func.col('Au.auid').alias('cited_authors')),
                     on = 'cited_document',
                     how = 'inner')
            .groupBy('cited_document','cited_doc_pub_year','cited_authors')
            .agg(func.countDistinct('Eid').alias('citation_count'))

            # h index window method
            .withColumn('auid', func.explode('cited_authors'))
            .withColumn('citation_rank_by_author', func.row_number().over(windowspec))
            .withColumn('rank_when_under_hgraph_line', func.when( func.col('citation_count') >= func.col('citation_rank_by_author'), func.col('citation_rank_by_author')))
            .groupBy('auid')
            .agg(func.max('rank_when_under_hgraph_line').alias('HIndex')) )

# now join against df_hyp2
df_hyp2 = df_hyp2.join(df_hindex, on='auid', how='left')

In [0]:
'''
# leave this as df_ani so you get all their publications
df_hindex = df_ani.withColumn('auid', explode ('Au.auid'))\
	.select('Eid', 'auid', 'Year', 'Title', 'CitationCount')\
	.join(df_authors, "auid", "right")\
	.groupBy("auid")\
	.agg(collect_list('CitationCount').alias('CitationsList'))\
	.withColumn('CitationsList', reverse(array_sort('CitationsList')))\
	.select('auid', posexplode('CitationsList').alias('CitationPosition', 'CitationCount'))\
	.filter(col('CitationCount') >= (col('CitationPosition') + 1))\
	.groupBy('auid')\
	.agg(max('CitationPosition').alias('HIndexMinus1'))\
	.withColumn('HIndex', col('HIndexMinus1') + 1)\
	.drop(col('HIndexMinus1'))
     
# display(df_hindex)

df_hyp2 = df_hyp2.join(df_hindex, "auid", "left")

# display(df_hyp2)'''

Out[14]: '\n# leave this as df_ani so you get all their publications\ndf_hindex = df_ani.withColumn(\'auid\', explode (\'Au.auid\'))\t.select(\'Eid\', \'auid\', \'Year\', \'Title\', \'CitationCount\')\t.join(df_authors, "auid", "right")\t.groupBy("auid")\t.agg(collect_list(\'CitationCount\').alias(\'CitationsList\'))\t.withColumn(\'CitationsList\', reverse(array_sort(\'CitationsList\')))\t.select(\'auid\', posexplode(\'CitationsList\').alias(\'CitationPosition\', \'CitationCount\'))\t.filter(col(\'CitationCount\') >= (col(\'CitationPosition\') + 1))\t.groupBy(\'auid\')\t.agg(max(\'CitationPosition\').alias(\'HIndexMinus1\'))\t.withColumn(\'HIndex\', col(\'HIndexMinus1\') + 1)\t.drop(col(\'HIndexMinus1\'))\n     \n# display(df_hindex)\n\ndf_hyp2 = df_hyp2.join(df_hindex, "auid", "left")\n\n# display(df_hyp2)'

In [0]:
windowspec = Window.partitionBy('auid').orderBy(func.col('nonselfcitationcount').desc())

df_hindex_no_sc =(df_ani
                  .select('eid', 
                          func.col('Au.auid').alias('citing_authors'),
                          func.col('Year').alias('year_of_citation'),
                          func.array_distinct('references').alias('references'))
                  .withColumn('cited_document', func.explode('references'))
                  .join(df_ani.select(func.col('Eid').alias('cited_document'),
                                      func.col('Year').alias('cited_doc_pub_year'),
                                      func.col('Au.auid').alias('cited_authors')),
                          on = 'cited_document',
                          how = 'inner')
                  .withColumn('self_citation_tf', arrays_overlap('citing_authors', 'cited_authors'))
                  .groupBy('cited_document','cited_doc_pub_year','cited_authors')
                  .agg(func.sum(
                    func.when(func.col('self_citation_tf'), func.lit(0)).otherwise(func.lit(1))).alias('nonselfcitationcount'))
                  
                  # h index window method
                  .withColumn('auid', func.explode('cited_authors'))
                  .withColumn('citation_rank_by_author', func.row_number().over(windowspec))
                  .withColumn('rank_when_under_hgraph_line', func.when( func.col('nonselfcitationcount') >= func.col('citation_rank_by_author'), func.col('citation_rank_by_author')))
                  .groupBy('auid')
                  .agg(func.max('rank_when_under_hgraph_line').alias('HIndex_noself')) )

# now join against df_hyp2
df_hyp2 = df_hyp2.join(df_hindex_no_sc, on='auid', how='left')

In [0]:
'''# no self citation h-index
df_from_corpus = (
  df_ani
  .select(col('eid').alias('from_eid'),
          col('year').alias('from_year'),
          col('Au.auid').alias('from_auids'), 
          size('Au').alias('from_n_Au'),
          col('CitationCount').alias('from_CC'), 
          'references') 
  .withColumn('references_to', explode('references'))
  .withColumnRenamed('year','from_year')
)

# metadata for everything that is cited
df_references_from_to = (
  df_from_corpus
  .join(df_ani
        # which fields to retain of the "to" publication, renaming for clarity
        .select(col('Au.auid').alias("to_auids"), col('year').alias('to_year'), col('eid'), size('Au').alias('to_n_A')),
        #join the "from" article's references to the "to" articles' IDs
        col('references_to') == df_ani['eid'], 'left')
        .withColumnRenamed('eid', 'to_eid')
        .withColumn('self_citation_tf', arrays_overlap('from_auids', 'to_auids'))
)

df_self_cc_counts = (
  df_references_from_to
  .groupBy('to_eid', 'to_auids') 
  .agg(count(lit(True)).alias('fullcitationcount'),
      sum(when(col('self_citation_tf'), lit(1)).otherwise(lit(0))).alias('selfcitationcount'),
      sum(when(col('self_citation_tf'), lit(0)).otherwise(lit(1))).alias('nonselfcitationcount'))
)

# display(df_self_cc_counts)

df_hindex = df_self_cc_counts.withColumn('auid', explode('to_auids'))\
	.groupBy("auid")\
	.agg(collect_list('nonselfcitationcount').alias('CitationsList'))\
	.withColumn('CitationsList', reverse(array_sort('CitationsList')))\
	.select ('auid', posexplode('CitationsList').alias('CitationPosition', 'nonselfcitationcount'))\
	.filter(col('nonselfcitationcount') >= (col('CitationPosition') + 1))\
	.groupBy('auid')\
	.agg(max('CitationPosition').alias('HIndexMinus1'))\
	.withColumn('HIndex_noself', col('HIndexMinus1') + 1)\
	.drop(col('HIndexMinus1'))
     
# display(df_hindex)

df_hyp2 = df_hyp2.join(df_hindex, "auid", "left")

# display(df_hyp2)'''


Out[16]: '# no self citation h-index\ndf_from_corpus = (\n  df_ani\n  .select(col(\'eid\').alias(\'from_eid\'),\n          col(\'year\').alias(\'from_year\'),\n          col(\'Au.auid\').alias(\'from_auids\'), \n          size(\'Au\').alias(\'from_n_Au\'),\n          col(\'CitationCount\').alias(\'from_CC\'), \n          \'references\') \n  .withColumn(\'references_to\', explode(\'references\'))\n  .withColumnRenamed(\'year\',\'from_year\')\n)\n\n# metadata for everything that is cited\ndf_references_from_to = (\n  df_from_corpus\n  .join(df_ani\n        # which fields to retain of the "to" publication, renaming for clarity\n        .select(col(\'Au.auid\').alias("to_auids"), col(\'year\').alias(\'to_year\'), col(\'eid\'), size(\'Au\').alias(\'to_n_A\')),\n        #join the "from" article\'s references to the "to" articles\' IDs\n        col(\'references_to\') == df_ani[\'eid\'], \'left\')\n        .withColumnRenamed(\'eid\', \'to_eid\')\n        .withColumn(\'self_citation_tf\', array

In [0]:
# create string index for Inferred Probable Gender 
indexer = StringIndexer(inputCol="Inferred_Probable_Gender", outputCol="IPG_index")
df_hyp2 = indexer.setHandleInvalid("skip").fit(df_hyp2).transform(df_hyp2)

# display(df_hyp2)

### Overall Summary 

For this research question, we will continue to use the model from the previous research question and add other article metrics to determine their contribution to the model. Our hypothesis is that each variable has an impact on the metrics variables (non-directional). Each author independent variable will be modeled separately and compared using confidence intervals as described above for which variable had the strongest impact on publication metrics. 

Using the linear model from the previous research question, we will add each author independent variable (affiliation country count, affiliation/institution count, h-index, gender, and gender ratio) in separate models to calculate their slope. We will determine if these slopes were important by examining if their confidence interval was greater than zero. We will then compare each slope to other slopes to see if their confidence intervals overlap. We have to examine the effects of each author metric separately. When examining first and/or last author information, the single author paper presents a unique problem because they are both first and last author. Therefore, these cannot be included in the same model. In order not to use the same variance twice, single author papers will be treated as first author only, and NA for last author analyses. If these were included in the same model, all single author papers would be dropped, which would bias the results.

In [0]:
# list of IVs for unique analyses 
independent_vars = ["country_count", "inst_count", "gender_percent", 
                   "HIndex", "HIndex_noself", "IPG_index", 
                   "2HIndex", "2HIndex_noself", "2IPG_index"]

# drop all the nulls in Y
df_hyp2 = df_hyp2.where(col("CitationCount").isNotNull())

In [0]:
df_hyp2.cache().display()

auid,Eid,Year,country_count,inst_count,ASJC,CitationCount,Authorseq,Inferred_Probable_Gender,Citations_5y,gender_percent,team_size,HIndex,HIndex_noself,IPG_index
6503849112,84943242123,2015,1,1,"List(2725, 2726)",35,7,male,19,0.5555555555555556,9,2,2,0.0
6503849112,85138229949,2022,1,3,List(2700),0,10,male,0,0.5384615384615384,13,2,2,0.0
6503849682,37986763,2003,1,1,List(2735),1,3,female,0,0.5,4,1,1,2.0
6503849682,84882280879,2013,1,3,List(2735),2,5,female,0,0.6666666666666666,6,1,1,2.0
6503850452,35381245,2001,1,1,List(2703),16,3,female,5,0.3333333333333333,6,5,5,2.0
6503850452,85081314502,2020,6,6,"List(2725, 2726)",117,38,female,129,0.22988505747126436,87,5,5,2.0
6503850452,85099369206,2021,3,4,"List(2725, 2726)",8,34,female,8,0.24444444444444444,90,5,5,2.0
6503850452,54549097198,2008,1,1,List(2700),0,16,female,0,0.6111111111111112,18,5,5,2.0
6503852954,85059535673,2018,1,2,"List(2211, 2500)",0,4,male,0,1.0,4,5,4,0.0
6503852954,70349974691,2009,1,2,"List(2210, 2211, 2500, 3104)",3,2,male,2,1.0,4,5,4,0.0


In [0]:
# loop over all the variables
for vars in independent_vars:
  # citations overall 
  # create dataframe, for these use unique Eids
  if (vars == "country_count") | (vars == "inst_count") | (vars == "gender_percent"):
    df_hyp2_overall = df_hyp2.dropDuplicates(['Eid'])
  elif (vars == "HIndex") | (vars == "HIndex_noself"):
    df_hyp2_overall = df_hyp2.filter((col("Authorseq") == 1))
  elif (vars == "IPG_index"):
    df_hyp2_overall = df_hyp2.filter((col("Authorseq") == 1))\
      .filter((col("Inferred_Probable_Gender") == "male") | (col("Inferred_Probable_Gender") == "female"))
  elif (vars == "2HIndex") | (vars == "2HIndex_noself"):
    df_hyp2_overall = df_hyp2.filter((col("Authorseq") == col("team_size")))
    vars = vars[1:]
  elif (vars == "2IPG_index"):
    df_hyp2_overall = df_hyp2.filter((col("Authorseq") == col("team_size")))\
      .filter((col("Inferred_Probable_Gender") == "male") | (col("Inferred_Probable_Gender") == "female"))
    vars = vars[1:]    

  # should drop nulls in X
  vectorAssembler = VectorAssembler(inputCols = ['Year', 'team_size', vars], outputCol = "pub_year")
  df_hyp2_overall = vectorAssembler.setHandleInvalid("skip").transform(df_hyp2_overall)
  df_hyp2_overall.cache()
  
  print(vars)
  print("\n----------\n")
  
  # put in the features X and prediction Y
  lr = LinearRegression(featuresCol = 'pub_year', labelCol = 'CitationCount')
  lr_model = lr.fit(df_hyp2_overall)
  model_output_df = modelsummary(lr_model, ("hyp2_CitationCount_"+vars), model_output_df)
  
  ## citations_5y
  # same DF just drop NULL citations_5y and newer articles 
  # if no citationcount then don't trust citations_5y
  df_hyp2_overall = df_hyp2_overall.where(col("Citations_5y").isNotNull())\
    .filter(col("Year") < 2017)
  
  # put in the features X and prediction Y
  lr = LinearRegression(featuresCol = 'pub_year', labelCol = 'Citations_5y')
  lr_model = lr.fit(df_hyp2_overall)
  model_output_df = modelsummary(lr_model, ("hyp2_Citation5y_"+vars), model_output_df)

  df_hyp2_overall.unpersist()



country_count

----------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##  -0.465274   0.002255 -206.294   0.000000
##  -0.022046   0.001588  -13.879   0.000000
##   7.458726   0.040440  184.439   0.000000
## 949.380812   4.517021  210.179   0.000000
## ---
## Mean squared error:  9875.393682 , RMSE:  99.375015
## Multiple R-squared: 0.006760 ,             Total iterations: 0
Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.221818   0.000750  295.861   0.000000
##   0.009434   0.000571   16.520   0.000000
##   4.461241   0.014662  304.281   0.000000
## -440.198178   1.497479 -293.959   0.000000
## ---
## Mean squared error:  641.953275 , RMSE:  25.336797
## Multiple R-squared: 0.031311 ,             Total iterations: 0
inst_count

----------

Note: the las

###Summary By Subject Area
Last, we will calculate each author independent variable separated by discipline and then visualize those slopes for each variable.

In [0]:
# explode
df_hyp2 = df_hyp2.withColumn("ASJC", explode("ASJC"))
df_hyp2 = df_hyp2.withColumn("ASJC", df_hyp2.ASJC.cast(IntegerType()))

df_hyp2 = df_hyp2\
  .withColumn('subjareas',
              when(df_hyp2.ASJC.between(1100, 1199), "Life Sciences")\
  .when(df_hyp2.ASJC.between(1200, 1299), "Social Sciences")\
  .when(df_hyp2.ASJC.between(1300, 1399), "Life Sciences")\
  .when(df_hyp2.ASJC.between(1400, 1499), "Social Sciences")\
  .when(df_hyp2.ASJC.between(1500, 1799), "Physical Sciences")\
  .when(df_hyp2.ASJC.between(1800, 1899), "Social Sciences")\
  .when(df_hyp2.ASJC.between(1900, 1999), "Physical Sciences")\
  .when(df_hyp2.ASJC.between(2000, 2099), "Social Sciences")\
  .when(df_hyp2.ASJC.between(2100, 2399), "Physical Sciences")\
  .when(df_hyp2.ASJC.between(2400, 2499), "Life Sciences")\
  .when(df_hyp2.ASJC.between(2500, 2699), "Physical Sciences")\
  .when(df_hyp2.ASJC.between(2700, 2799), "Health Sciences")\
  .when(df_hyp2.ASJC.between(2800, 2899), "Life Sciences")\
  .when(df_hyp2.ASJC.between(2900, 2999), "Health Sciences")\
  .when(df_hyp2.ASJC.between(3000, 3099), "Life Sciences")\
  .when(df_hyp2.ASJC.between(3100, 3199), "Physical Sciences")\
  .when(df_hyp2.ASJC.between(3200, 3399), "Social Sciences")\
  .when(df_hyp2.ASJC.between(3400, 3699), "Health Sciences")\
  .otherwise("Multidisciplinary")
  )

In [0]:
for AREA in descriptives_output_df.subjareas.unique():
  if AREA == 'undefined': continue
  print(AREA)
  print("\n-------\n")

  # loop over all the variables
  for vars in independent_vars:
    # citations overall 
    # create dataframe, for these use unique Eids
    if (vars == "country_count") | (vars == "inst_count") | (vars == "gender_percent"):
      df_hyp2_overall = df_hyp2.dropDuplicates(['Eid'])
    elif (vars == "HIndex") | (vars == "HIndex_noself"):
      df_hyp2_overall = df_hyp2.filter((col("Authorseq") == 1))
    elif (vars == "IPG_index"):
      df_hyp2_overall = df_hyp2.filter((col("Authorseq") == 1))\
        .filter((col("Inferred_Probable_Gender") == "male") | (col("Inferred_Probable_Gender") == "female"))
    elif (vars == "2HIndex") | (vars == "2HIndex_noself"):
      df_hyp2_overall = df_hyp2.filter((col("Authorseq") == col("team_size")))
      vars = vars[1:]
    elif (vars == "2IPG_index"):
      df_hyp2_overall = df_hyp2.filter((col("Authorseq") == col("team_size")))\
        .filter((col("Inferred_Probable_Gender") == "male") | (col("Inferred_Probable_Gender") == "female"))
      vars = vars[1:]    

    # also filter for subject area 
    df_hyp2_overall = df_hyp2_overall.filter(col("subjareas") == AREA)
    
    # should drop nulls in X
    vectorAssembler = VectorAssembler(inputCols = ['Year', 'team_size', vars], outputCol = "pub_year")
    df_hyp2_overall = vectorAssembler.setHandleInvalid("skip").transform(df_hyp2_overall)

    df_hyp2_overall.cache()

    print(vars)
    print("\n----------\n")

    # put in the features X and prediction Y
    lr = LinearRegression(featuresCol = 'pub_year', labelCol = 'CitationCount')
    lr_model = lr.fit(df_hyp2_overall)
    model_output_df = modelsummary(lr_model, ("hyp2_CitationCount_"+vars+"_"+AREA), model_output_df)

    ## citations_5y
    # same DF just drop NULL citations_5y and newer articles 
    # if no citationcount then don't trust citations_5y
    df_hyp2_overall = df_hyp2_overall.where(col("Citations_5y").isNotNull())\
      .filter(col("Year") < 2017)

    # put in the features X and prediction Y
    lr = LinearRegression(featuresCol = 'pub_year', labelCol = 'Citations_5y')
    lr_model = lr.fit(df_hyp2_overall)
    model_output_df = modelsummary(lr_model, ("hyp2_Citation5y_"+vars+"_"+AREA), model_output_df)
    df_hyp2_overall.unpersist()


# Output Tables

Output tables for writing manuscript in R markdown.

In [0]:
# write out results section until I can get download link
pd.set_option('display.max_rows', None)
model_output_df

,model,coef,se,t,p,r2
0,hyp2_CitationCount_country_count,-0.465274,0.002255,-206.294193,0.0,0.00676
1,hyp2_CitationCount_country_count,-0.022046,0.001588,-13.878923,0.0,0.00676
2,hyp2_CitationCount_country_count,7.458726,0.04044,184.438618,0.0,0.00676
3,hyp2_CitationCount_country_count,949.380812,4.517021,210.17851,0.0,0.00676
4,hyp2_Citation5y_country_count,0.221818,0.00075,295.861074,0.0,0.031311
5,hyp2_Citation5y_country_count,0.009434,0.000571,16.519718,0.0,0.031311
6,hyp2_Citation5y_country_count,4.461241,0.014662,304.280577,0.0,0.031311
7,hyp2_Citation5y_country_count,-440.198178,1.497479,-293.959494,0.0,0.031311
8,hyp2_CitationCount_inst_count,-0.492492,0.002281,-215.950153,0.0,0.006752
9,hyp2_CitationCount_inst_count,-0.156769,0.002031,-77.182576,0.0,0.006752


In [0]:
descriptives_output_df

,model,subjareas,mean,sd,sample,median,iqr_25,iqr_75


EXECUTION DETAILS:
Spark version: 12.2.x-aarch64-scala2.12, 
Cluster Node Type: r6gd.2xlarge, 
Driver Node Type: r6gd.2xlarge, 
Worker Node Type: r6gd.2xlarge


DATA SNAPSHOT DETAILS:
This notebook uses the latest snapshot date for this version: "v008.20230509104116/", (09 May 2023)

 The following are the Scopus snapshot dates available for datasets at v008:
['v008.20220314092239/', 'v008.20220906063204/', 'v008.20221021151538/', 'v008.20230509104116/']
(If you want to use a different snapshot, in the first cell of your notebook, set the variable `custom_snapshot_date` to one of the options listed above (string data type))



DATA SAMPLE DETAILS:
Setting paths to run on full datasets (100% Scopus publication volume)


In [0]:
model_output_df.to_csv('/dbfs/Projects/proj_050_teamsci/model_output_200723_part2.csv')
descriptives_output_df.to_csv('/dbfs/Projects/proj_050_teamsci/descriptives_output_df_200723_part2.csv')

In [0]:
%fs ls s3a://elsevier-fcads-icsrlab-output/shared/proj_050_teamsci

path,name,size,modificationTime
s3a://elsevier-fcads-icsrlab-output/shared/proj_050_teamsci/descriptives_output_df_200723_part2.csv,descriptives_output_df_200723_part2.csv,53,1689933341000
s3a://elsevier-fcads-icsrlab-output/shared/proj_050_teamsci/model_output_200723_part2.csv,model_output_200723_part2.csv,8325,1689933305000
